In [1]:
import tkinter as tk
from tkinter import filedialog, ttk, messagebox
from tkcalendar import DateEntry
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
import numpy as np
import warnings
import time

warnings.filterwarnings("ignore")

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive.readonly"
]

UTM_VALUES = [
    "bik","cem","crm","crn","crk",
    "kwikchat","kwikengage","sms",
    "tcwaycrm","wa","whatsapp",
    "whatsappordershipped"
]

# ================= DEFAULT LINKS =================
DEFAULT_MTD_LINK = "https://docs.google.com/spreadsheets/d/1-n_dg-d1wAgNtZxk3hOyPQCS2SGY8nYFnITNFbyQ-lw/edit?gid=779320564#gid=779320564"

DEFAULT_SPEND_LINK = "https://docs.google.com/spreadsheets/d/1jixKAjLP0zcLec1YoBV2Zlwhj9jP7wBiM7Xb03-pRUw/edit?gid=0#gid=0"

# ================= BROWSE =================
def browse_file(entry):

    file = filedialog.askopenfilename()

    entry.delete(0, tk.END)
    entry.insert(0, file)

# ================= CLEAN HEADERS =================
def clean_headers(df):

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return df

# ================= CLEAN UTM =================
def clean_utm(series):

    return (
        series.astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r'\s+', '', regex=True)
        .str.replace(r'[^a-z0-9]', '', regex=True)
    )

# ================= LOAD SPEND =================
def load_spend_df(ws):

    data = ws.get_all_values()

    headers = [
        str(h).strip().lower()
        for h in data[2]
    ]

    df = pd.DataFrame(
        data[3:],
        columns=headers
    )

    df = df.loc[:, ~df.columns.duplicated()]

    return df

# ================= CALCULATE SPEND =================
def calculate_spend(df, start, end):

    df["date"] = pd.to_datetime(
        df["date"],
        errors='coerce',
        dayfirst=True
    ).dt.date

    df = df.dropna(subset=["date"])

    start = start.date()
    end = end.date()

    month_str = start.strftime("%b'%y")

    df = df[
        df["month"]
        .astype(str)
        .str.strip() == month_str
    ]

    df = df[
        (df["date"] >= start)
        & (df["date"] <= end)
    ]

    crm = (
        df["crm"]
        .astype(str)
        .str.replace(r'[₹,]', '', regex=True)
        .str.strip()
    )

    crm = pd.to_numeric(
        crm,
        errors='coerce'
    )

    return round(float(crm.sum()), 1)

# ================= PROCESS GOKWIK =================
def process_gokwik(csv, start, end):

    df = pd.read_csv(csv)

    df = clean_headers(df)

    df["created at"] = pd.to_datetime(
        df["created at"],
        errors='coerce',
        dayfirst=True
    ).dt.date

    df = df.dropna(subset=["created at"])

    start = start.date()
    end = end.date()

    df = df[
        (df["created at"] >= start)
        & (df["created at"] <= end)
    ]

    utm_col = [
        c for c in df.columns
        if "utm" in c
    ][0]

    df = df[
        clean_utm(df[utm_col])
        .isin(UTM_VALUES)
    ]

    return round(
        pd.to_numeric(
            df["grand total"],
            errors='coerce'
        ).sum(),
        1
    )

# ================= FIND BLOCK =================
def find_block(sheet, brand, end_date):

    target = (
        f"{brand} - {end_date.strftime('%b')}"
        .lower()
    )

    data = sheet.get_all_values()

    for i, row in enumerate(data):

        if any(
            str(c).strip().lower() == target
            for c in row
        ):
            return i + 1

    raise Exception("Month block not found")

# ================= RUN =================
def run_script():

    try:

        status_var.set("Running...")

        creds = Credentials.from_service_account_file(
            "services_account.json",
            scopes=SCOPES
        )

        client = gspread.authorize(creds)

        brand = brand_var.get()

        start = pd.to_datetime(
            start_cal.get_date()
        )

        end = pd.to_datetime(
            end_cal.get_date()
        )

        # ================= SPEND TAB MAP =================
        tab_map = {
            "AVON": "Avon",
            "ENAMOR": "Enamor",
            "NEUROGUM": "Neuro"
        }

        target_spend_tab = tab_map.get(
            brand,
            brand
        )

        # ================= LINKS =================
        final_mtd_link = (
            mtd_link.get()
            if change_mtd_var.get()
            else DEFAULT_MTD_LINK
        )

        final_spend_link = (
            spend_link.get()
            if change_spend_var.get()
            else DEFAULT_SPEND_LINK
        )

        headless_cur = round(
            float(headless_current.get() or 0),
            1
        )

        headless_prev = round(
            float(headless_previous.get() or 0),
            1
        )

        sheet = (
            client
            .open_by_url(final_mtd_link)
            .worksheet(brand)
        )

        spend_ws = (
            client
            .open_by_url(final_spend_link)
            .worksheet(target_spend_tab)
        )

        spend_df = load_spend_df(spend_ws)

        utm = process_gokwik(
            current_file.get(),
            start,
            end
        )

        total = round(
            headless_cur + utm,
            1
        )

        spend = calculate_spend(
            spend_df,
            start,
            end
        )

        roas = (
            round(total / spend, 1)
            if spend else 0.0
        )

        # ================= PREVIOUS =================
        prev_start = start - pd.DateOffset(months=1)
        prev_end = end - pd.DateOffset(months=1)

        utm_prev = process_gokwik(
            prev_file.get(),
            prev_start,
            prev_end
        )

        total_prev = round(
            headless_prev + utm_prev,
            1
        )

        spend_prev = calculate_spend(
            spend_df,
            prev_start,
            prev_end
        )

        roas_prev = (
            round(total_prev / spend_prev, 1)
            if spend_prev else 0.0
        )

        row_idx = find_block(
            sheet,
            brand,
            end
        )

        # ================= HISTORY SHIFT =================
        history_start_col = 6

        for col in range(
            20,
            history_start_col,
            -1
        ):

            col_letter = chr(64 + col)

            prev_col_letter = chr(
                64 + col - 1
            )

            values = sheet.get(
                f"{prev_col_letter}{row_idx+4}:{prev_col_letter}{row_idx+10}"
            )

            if values:

                sheet.update(
                    f"{col_letter}{row_idx+4}",
                    values
                )

        old_header = sheet.acell(
            f"B{row_idx+4}"
        ).value

        old_values = sheet.get(
            f"B{row_idx+5}:B{row_idx+10}"
        )

        if old_header:

            sheet.update(
                f"F{row_idx+4}",
                [[old_header]]
            )

            sheet.update(
                f"F{row_idx+5}",
                old_values
            )

        # ================= CURRENT =================
        sheet.update(
            f"B{row_idx+4}",
            [[f"Till {end.day}th"]]
        )

        sheet.update(
            f"B{row_idx+5}",
            [[round(headless_cur, 1)]]
        )

        sheet.update(
            f"B{row_idx+6}",
            [[round(utm, 1)]]
        )

        sheet.update(
            f"B{row_idx+8}",
            [[round(total, 1)]]
        )

        sheet.update(
            f"B{row_idx+9}",
            [[round(spend, 1)]]
        )

        sheet.update(
            f"B{row_idx+10}",
            [[round(roas, 1)]]
        )

        # ================= PREVIOUS =================
        sheet.update(
            f"E{row_idx+5}",
            [[round(headless_prev, 1)]]
        )

        sheet.update(
            f"E{row_idx+6}",
            [[round(utm_prev, 1)]]
        )

        sheet.update(
            f"E{row_idx+8}",
            [[round(total_prev, 1)]]
        )

        sheet.update(
            f"E{row_idx+9}",
            [[round(spend_prev, 1)]]
        )

        sheet.update(
            f"E{row_idx+10}",
            [[round(roas_prev, 1)]]
        )

        # ================= FORMULAS =================
        total_row = row_idx + 8
        spend_row = row_idx + 9
        target_row = row_idx + 2

        day = end.day
        days = pd.Timestamp(end).days_in_month

        sheet.update(
            f"C{total_row}",
            [[""]]
        )

        sheet.update(
            f"C{spend_row}",
            [[""]]
        )

        time.sleep(1)

        sheet.update(
            f"C{total_row}",
            [[f"=(C{target_row}/{days})*{day}"]],
            value_input_option="USER_ENTERED"
        )

        sheet.update(
            f"C{spend_row}",
            [[f"=(A{target_row}/{days})*{day}"]],
            value_input_option="USER_ENTERED"
        )

        sheet.update(
            f"D{total_row}",
            [[f"=IFERROR(B{total_row}/C{total_row},0)"]],
            value_input_option="USER_ENTERED"
        )

        sheet.update(
            f"D{spend_row}",
            [[f"=IFERROR(B{spend_row}/C{spend_row},0)"]],
            value_input_option="USER_ENTERED"
        )

        status_var.set("Done ✅")

        messagebox.showinfo(
            "Success",
            "MTD Updated Successfully"
        )

    except Exception as e:

        status_var.set("Error ❌")

        messagebox.showerror(
            "Error",
            str(e)
        )

# ================= UI =================
root = tk.Tk()

root.title("MTD CRM Automation")
root.geometry("750x650")

frame = tk.Frame(root)
frame.pack(padx=20, pady=20)

# ================= ROW UI =================
def row_ui(label, var, browse=False):

    f = tk.Frame(frame)
    f.pack(fill="x", pady=5)

    tk.Label(
        f,
        text=label,
        width=20
    ).pack(side="left")

    e = tk.Entry(
        f,
        textvariable=var,
        width=40
    )

    e.pack(side="left")

    if browse:

        tk.Button(
            f,
            text="Browse",
            command=lambda: browse_file(e)
        ).pack(side="left")

    return f

# ================= VARIABLES =================
brand_var = tk.StringVar(value="AVON")

current_file = tk.StringVar()
prev_file = tk.StringVar()

headless_current = tk.StringVar()
headless_previous = tk.StringVar()

mtd_link = tk.StringVar()
spend_link = tk.StringVar()

status_var = tk.StringVar(value="Idle")

change_mtd_var = tk.BooleanVar(value=False)
change_spend_var = tk.BooleanVar(value=False)

# ================= BRAND =================
ttk.Combobox(
    frame,
    textvariable=brand_var,
    values=["AVON", "NEUROGUM", "ENAMOR"]
).pack(pady=5)

# ================= START DATE =================
start_frame = tk.Frame(frame)
start_frame.pack(pady=3)

tk.Label(
    start_frame,
    text="Start Date",
    width=15,
    anchor="w"
).pack(side="left")

start_cal = DateEntry(
    start_frame,
    width=12
)

start_cal.pack(side="left")

# ================= END DATE =================
end_frame = tk.Frame(frame)
end_frame.pack(pady=3)

tk.Label(
    end_frame,
    text="End Date",
    width=15,
    anchor="w"
).pack(side="left")

end_cal = DateEntry(
    end_frame,
    width=12
)

end_cal.pack(side="left")

# ================= FILES =================
row_ui(
    "Current CSV",
    current_file,
    True
)

row_ui(
    "Previous CSV",
    prev_file,
    True
)

row_ui(
    "Headless Current",
    headless_current
)

row_ui(
    "Headless Previous",
    headless_previous
)

# ================= MTD LINK =================
tk.Checkbutton(
    frame,
    text="Change MTD Sheet Link?",
    variable=change_mtd_var,
    command=lambda:
    mtd_row.pack(fill="x")
    if change_mtd_var.get()
    else mtd_row.pack_forget()
).pack(anchor="w")

mtd_row = row_ui(
    "New MTD Link",
    mtd_link
)

mtd_row.pack_forget()

# ================= SPEND LINK =================
tk.Checkbutton(
    frame,
    text="Change Spend Sheet Link?",
    variable=change_spend_var,
    command=lambda:
    spend_row.pack(fill="x")
    if change_spend_var.get()
    else spend_row.pack_forget()
).pack(anchor="w")

spend_row = row_ui(
    "New Spend Link",
    spend_link
)

spend_row.pack_forget()

# ================= RUN BUTTON =================
tk.Button(
    frame,
    text="RUN",
    command=run_script,
    bg="#28a745",
    fg="white",
    font=('Arial', 10, 'bold')
).pack(pady=20)

# ================= STATUS =================
tk.Label(
    frame,
    textvariable=status_var
).pack()

root.mainloop()
